# sprint 5. robustez, feature engineering e comparação de modelos

projeto: infant dysbiosis predictor  
sprint: 12/5  

objetivos:
1. **feature engineering** — explorar colunas não utilizadas (`State`, `baby_birthday`) e extrair novas features (mês, estação de nascimento)
2. **comparação multi-modelo** — Logistic Regression, SVM, Decision Tree, Random Forest, Gradient Boosting, XGBoost
3. **ablação de features** — impacto de remover AMR e VF no recall e AUC
4. **C2 vs C1+C3** — modelo binário dedicado ao cluster intermediário
5. **exportar todos os resultados em JSON centralizado**

> pre-requisito: sprints 1–4 executadas (`data/processed/` populado)

## 0. imports

In [ ]:
import subprocess, sys, importlib
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'xgboost', 'scikit-learn', 'shap', 'openpyxl', '-q'], check=True)
importlib.invalidate_caches()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 30)
SEED = 42
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print('imports ok')

## 1. feature engineering

na sprint 1 foram descartadas `baby_birthday` e `State` por não serem features clínicas diretas.  
aqui revisitamos essas colunas para ver se delas é possível extrair sinal útil:  
- `baby_birthday` → mês de nascimento e estação do ano (sazonalidade afeta colonização microbiana?)
- `State` → estado de residência (proxy de exposição ambiental, dieta regional)

In [ ]:
# carregar Data 2 bruto
df2_raw = pd.read_excel('../data/Supplementary Data 2.xlsx')
clinical_cols = list(df2_raw.columns[:14])

print('colunas clinicas disponiveis (Data 2):')
for col in clinical_cols:
    n_miss = df2_raw[col].isnull().sum()
    n_uniq = df2_raw[col].nunique()
    dtype  = str(df2_raw[col].dtype)
    print(f'  {col:<45} dtype={dtype:<15} unicos={n_uniq:<5} missing={n_miss}')

print()
print('=== State ===')
print(df2_raw['State'].value_counts().to_string())

print()
print('=== baby_birthday (primeiras 5 linhas) ===')
print(df2_raw['baby_birthday'].head())

In [ ]:
# extrair features de baby_birthday e State
df2_feat = df2_raw[['sample_id', 'State', 'baby_birthday']].copy()
df2_feat['baby_birthday'] = pd.to_datetime(df2_feat['baby_birthday'], errors='coerce')

# mes e estacao
df2_feat['birth_month'] = df2_feat['baby_birthday'].dt.month

def month_to_season(m):
    if pd.isna(m): return 'unknown'
    m = int(m)
    if m in [12, 1, 2]: return 'winter'
    elif m in [3, 4, 5]: return 'spring'
    elif m in [6, 7, 8]: return 'summer'
    return 'fall'

df2_feat['birth_season'] = df2_feat['birth_month'].map(month_to_season)

# State: top 4 estados + 'Other'
top_states = df2_feat['State'].value_counts().head(4).index
df2_feat['state_grouped'] = df2_feat['State'].apply(
    lambda x: x if x in top_states else 'Other'
)

# normalizar birth_month
scaler_month = StandardScaler()
month_vals = df2_feat['birth_month'].fillna(df2_feat['birth_month'].median())
df2_feat['birth_month_scaled'] = scaler_month.fit_transform(month_vals.values.reshape(-1, 1)).ravel()

print('distribuicao de estacoes:')
print(df2_feat['birth_season'].value_counts())
print()
print('distribuicao de estados (agrupados):')
print(df2_feat['state_grouped'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df2_feat['birth_season'].value_counts().plot(kind='bar', ax=axes[0], color='#378ADD')
axes[0].set_title('nascimentos por estacao')
axes[0].tick_params(axis='x', rotation=0)
df2_feat['state_grouped'].value_counts().plot(kind='bar', ax=axes[1], color='#1D9E75')
axes[1].set_title('nascimentos por estado (agrupado)')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# construir dataset estendido (encoded + novas features)
df_m1_enc = pd.read_csv('../data/processed/model1_encoded.csv')
df_m2_enc = pd.read_csv('../data/processed/model2_encoded.csv')

# OHE para estacao e estado
season_dummies = pd.get_dummies(df2_feat['birth_season'], prefix='season').astype(int)
state_dummies  = pd.get_dummies(df2_feat['state_grouped'], prefix='state').astype(int)
new_feat_df = pd.concat([
    df2_feat[['sample_id', 'birth_month_scaled']],
    season_dummies,
    state_dummies
], axis=1)

NEW_FEAT_COLS = ['birth_month_scaled'] + list(season_dummies.columns) + list(state_dummies.columns)

df_m1_ext = df_m1_enc.merge(new_feat_df, on='sample_id', how='left').fillna(0)
df_m2_ext = df_m2_enc.merge(new_feat_df, on='sample_id', how='left').fillna(0)

print(f'novas features ({len(NEW_FEAT_COLS)}): {NEW_FEAT_COLS}')
print(f'dataset M1 estendido: {df_m1_ext.shape}')
print(f'dataset M2 estendido: {df_m2_ext.shape}')

## 2. modelo 1: comparação multi-modelo (desfecho adverso)

6 modelos avaliados com CV estratificada 5-fold nas mesmas features e mesma métrica (recall, ROC-AUC, F1).  
todos os modelos usam `class_weight='balanced'` ou equivalente para lidar com o desbalanceamento (~30% positivos).  
features: as 12 do modelo 1 (inclui `antibiotics_first_2_years`).

In [ ]:
TARGET_M1 = 'adverse_outcomes'
FEAT_M1   = [c for c in df_m1_enc.columns if c not in ('sample_id', TARGET_M1)]

X_m1 = df_m1_enc[FEAT_M1].values
y_m1 = df_m1_enc[TARGET_M1].values

spw_m1 = float((y_m1 == 0).sum() / (y_m1 == 1).sum())

MODELS_M1 = {
    'Logistic Regression': LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=1000, random_state=SEED),
    'SVM linear': SVC(
        kernel='linear', C=1.0, class_weight='balanced',
        probability=True, random_state=SEED),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5, class_weight='balanced', random_state=SEED),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced', random_state=SEED, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1, random_state=SEED),
    'XGBoost': XGBClassifier(
        n_estimators=400, max_depth=3, learning_rate=0.05,
        scale_pos_weight=spw_m1, eval_metric='logloss',
        random_state=SEED, n_jobs=-1, verbosity=0),
}

print(f'target: {TARGET_M1}  |  n={len(y_m1)}  |  positivos: {y_m1.sum()} ({y_m1.mean()*100:.1f}%)')
print(f'features ({len(FEAT_M1)}): {FEAT_M1}')
print(f'scale_pos_weight (XGBoost): {spw_m1:.2f}')
print(f'modelos a comparar: {list(MODELS_M1)}')

In [ ]:
SCORING_M1 = {'recall': 'recall', 'roc_auc': 'roc_auc', 'f1': 'f1'}

results_m1 = {}
for name, model in MODELS_M1.items():
    cv_res = cross_validate(model, X_m1, y_m1, cv=cv5,
                            scoring=SCORING_M1, n_jobs=-1)
    results_m1[name] = {
        metric: {'mean': float(cv_res[f'test_{metric}'].mean()),
                 'std' : float(cv_res[f'test_{metric}'].std())}
        for metric in SCORING_M1
    }
    print(f'  {name:<22}  recall={results_m1[name]["recall"]["mean"]:.3f}'
          f'  auc={results_m1[name]["roc_auc"]["mean"]:.3f}'
          f'  f1={results_m1[name]["f1"]["mean"]:.3f}')

print('\nconcluido')

In [ ]:
metrics_order = ['recall', 'roc_auc', 'f1']
models_order  = list(results_m1)

means_m1 = pd.DataFrame(
    {m: [results_m1[mod][m]['mean'] for mod in models_order] for m in metrics_order},
    index=models_order
)
stds_m1 = pd.DataFrame(
    {m: [results_m1[mod][m]['std'] for mod in models_order] for m in metrics_order},
    index=models_order
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# heatmap
sns.heatmap(means_m1, annot=True, fmt='.3f', cmap='YlGn', ax=axes[0],
            linewidths=0.5, vmin=0, vmax=1, annot_kws={'size': 11})
axes[0].set_title('media CV 5-fold — Modelo 1 (desfecho adverso)', fontsize=11)
axes[0].set_xlabel('')

# barras grouped por metrica
x = np.arange(len(models_order))
w = 0.25
colors = ['#E24B4A', '#378ADD', '#1D9E75']
for i, (metric, color) in enumerate(zip(metrics_order, colors)):
    vals = means_m1[metric].values
    errs = stds_m1[metric].values
    axes[1].bar(x + i*w, vals, w, label=metric, color=color,
                alpha=0.85, yerr=errs, capsize=3)
axes[1].set_xticks(x + w)
axes[1].set_xticklabels(models_order, rotation=25, ha='right', fontsize=9)
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('score (CV 5-fold)')
axes[1].set_title('comparacao por modelo — Modelo 1', fontsize=11)
axes[1].legend()

plt.suptitle('Modelo 1: comparacao multi-modelo (desfecho adverso, n=208)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('\nranking por recall:')
print(means_m1['recall'].sort_values(ascending=False).round(3).to_string())

## 3. ablação de features: modelo 1

quanto o desempenho cai quando retiramos as features de microbioma (AMR e VF)?  
subsets testados com XGBoost (melhor modelo da sprint 3):

| subset | features removidas |
|---|---|
| todas (baseline) | — |
| sem AMR | `amr_gene_sum_abundance` |
| sem VF | `vf_sum_abundance` |
| sem AMR + VF | ambas |
| clinicas (sem AMR/VF/antibiotics) | AMR, VF e `antibiotics_first_2_years` |
| + features novas | + mes, estacao, estado |

In [ ]:
FEAT_M1_EXT = [c for c in df_m1_ext.columns if c not in ('sample_id', TARGET_M1)]

SUBSETS_M1 = {
    'todas (baseline)'           : FEAT_M1,
    'sem amr_gene'               : [f for f in FEAT_M1 if f != 'amr_gene_sum_abundance'],
    'sem vf_sum'                 : [f for f in FEAT_M1 if f != 'vf_sum_abundance'],
    'sem amr + vf'               : [f for f in FEAT_M1 if f not in
                                    ('amr_gene_sum_abundance', 'vf_sum_abundance')],
    'clinicas (sem amr/vf/ab)'   : [f for f in FEAT_M1 if f not in
                                    ('amr_gene_sum_abundance', 'vf_sum_abundance',
                                     'antibiotics_first_2_years')],
    '+ features novas'           : FEAT_M1_EXT,
}

ablation_results = {}
for subset_name, feats in SUBSETS_M1.items():
    if subset_name == '+ features novas':
        X_sub = df_m1_ext[feats].values
    else:
        X_sub = df_m1_enc[feats].values

    spw_sub = float((y_m1 == 0).sum() / (y_m1 == 1).sum())
    xgb = XGBClassifier(
        n_estimators=400, max_depth=3, learning_rate=0.05,
        scale_pos_weight=spw_sub, eval_metric='logloss',
        random_state=SEED, n_jobs=-1, verbosity=0
    )
    cv_res = cross_validate(xgb, X_sub, y_m1, cv=cv5,
                            scoring=SCORING_M1, n_jobs=-1)
    ablation_results[subset_name] = {
        metric: {'mean': float(cv_res[f'test_{metric}'].mean()),
                 'std' : float(cv_res[f'test_{metric}'].std())}
        for metric in SCORING_M1
    }
    print(f'  {subset_name:<30} n_feat={len(feats):>2}'
          f'  recall={ablation_results[subset_name]["recall"]["mean"]:.3f}'
          f'  auc={ablation_results[subset_name]["roc_auc"]["mean"]:.3f}'
          f'  f1={ablation_results[subset_name]["f1"]["mean"]:.3f}')

In [ ]:
subsets_order = list(ablation_results)
abl_means = pd.DataFrame(
    {m: [ablation_results[s][m]['mean'] for s in subsets_order] for m in metrics_order},
    index=subsets_order
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(abl_means, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0],
            linewidths=0.5, vmin=0, vmax=1, annot_kws={'size': 10})
axes[0].set_title('ablacao de features — XGBoost (Modelo 1)', fontsize=11)

# delta em relacao ao baseline
baseline = abl_means.loc['todas (baseline)']
delta = abl_means.subtract(baseline)
sns.heatmap(delta, annot=True, fmt='+.3f', cmap='RdYlGn', ax=axes[1],
            linewidths=0.5, center=0, vmin=-0.3, vmax=0.3, annot_kws={'size': 10})
axes[1].set_title('delta vs baseline (verde = melhora)', fontsize=11)

plt.suptitle('impacto de remover features — XGBoost, Modelo 1', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('\ndelta recall ao remover AMR + VF:')
baseline_recall = abl_means.loc['todas (baseline)', 'recall']
noclin_recall   = abl_means.loc['sem amr + vf', 'recall']
print(f'  baseline     : {baseline_recall:.3f}')
print(f'  sem amr + vf : {noclin_recall:.3f}')
print(f'  delta        : {noclin_recall - baseline_recall:+.3f}')

## 4. modelo 2: C2 vs C1+C3 (problema binário)

na sprint 4, o SHAP mostrou que C2 tem os valores mais baixos e difusos — o modelo não sabe como identificá-lo.  
aqui formulamos um problema binário: **C2 = 1, C1+C3 = 0** e treinamos todos os 6 modelos.  
objetivo: descobrir se algum modelo consegue separar C2 dos demais, e quais features são mais discriminativas.

In [ ]:
TARGET_M2 = 'dmm_clusters_taxa'
FEAT_M2   = [c for c in df_m2_enc.columns if c not in ('sample_id', TARGET_M2)]
X_m2      = df_m2_enc[FEAT_M2].values
y_m2      = df_m2_enc[TARGET_M2].values

# target binario: C2 vs outros
y_c2 = (y_m2 == 'C2').astype(int)
spw_c2 = float((y_c2 == 0).sum() / (y_c2 == 1).sum())

MODELS_C2 = {
    'Logistic Regression': LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=1000, random_state=SEED),
    'SVM linear': SVC(
        kernel='linear', C=1.0, class_weight='balanced',
        probability=True, random_state=SEED),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5, class_weight='balanced', random_state=SEED),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced', random_state=SEED, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1, random_state=SEED),
    'XGBoost': XGBClassifier(
        n_estimators=400, max_depth=3, learning_rate=0.05,
        scale_pos_weight=spw_c2, eval_metric='logloss',
        random_state=SEED, n_jobs=-1, verbosity=0),
}

print(f'C2 vs C1+C3  |  n={len(y_c2)}  |  C2: {y_c2.sum()} ({y_c2.mean()*100:.1f}%)')

results_c2 = {}
for name, model in MODELS_C2.items():
    cv_res = cross_validate(model, X_m2, y_c2, cv=cv5,
                            scoring=SCORING_M1, n_jobs=-1)
    results_c2[name] = {
        metric: {'mean': float(cv_res[f'test_{metric}'].mean()),
                 'std' : float(cv_res[f'test_{metric}'].std())}
        for metric in SCORING_M1
    }
    print(f'  {name:<22}  recall={results_c2[name]["recall"]["mean"]:.3f}'
          f'  auc={results_c2[name]["roc_auc"]["mean"]:.3f}'
          f'  f1={results_c2[name]["f1"]["mean"]:.3f}')

In [ ]:
means_c2 = pd.DataFrame(
    {m: [results_c2[mod][m]['mean'] for mod in results_c2] for m in metrics_order},
    index=list(results_c2)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(means_c2, annot=True, fmt='.3f', cmap='Blues', ax=axes[0],
            linewidths=0.5, vmin=0, vmax=1, annot_kws={'size': 11})
axes[0].set_title('C2 vs C1+C3 — comparacao multi-modelo', fontsize=11)

# feature importance do melhor modelo (XGBoost)
xgb_c2 = XGBClassifier(
    n_estimators=400, max_depth=3, learning_rate=0.05,
    scale_pos_weight=spw_c2, eval_metric='logloss',
    random_state=SEED, n_jobs=-1, verbosity=0
)
xgb_c2.fit(X_m2, y_c2)

imp_c2 = pd.Series(xgb_c2.feature_importances_, index=FEAT_M2).sort_values()
colors_imp = ['#E24B4A' if v == imp_c2.max() else '#378ADD' for v in imp_c2.values]
imp_c2.plot(kind='barh', ax=axes[1], color=colors_imp, alpha=0.85)
axes[1].set_title('feature importance XGBoost — C2 vs C1+C3', fontsize=11)
axes[1].set_xlabel('importance (gain)')
for i, v in enumerate(imp_c2.values):
    axes[1].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=8)

plt.suptitle('C2 vs C1+C3: separabilidade do cluster intermediario', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

best_c2_model = means_c2['roc_auc'].idxmax()
print(f'melhor modelo (AUC): {best_c2_model}  →  AUC={means_c2.loc[best_c2_model,"roc_auc"]:.3f}')
print()
print('feature importance XGBoost (C2 vs outros), ranking:')
for rank, (feat, val) in enumerate(imp_c2.sort_values(ascending=False).items(), 1):
    print(f'  {rank:>2}. {feat:<45} {val:.4f}')

In [ ]:
# SHAP para o modelo C2 binario
explainer_c2 = shap.TreeExplainer(xgb_c2)
sv_c2 = explainer_c2.shap_values(X_m2)
if isinstance(sv_c2, list): sv_c2 = sv_c2[1]

shap.summary_plot(sv_c2, X_m2, feature_names=FEAT_M2, plot_type='bar', show=False)
plt.gcf().set_size_inches(8, 5)
plt.title('SHAP — importancia global: C2 vs C1+C3 (XGBoost)', fontsize=11)
plt.tight_layout()
plt.show()

shap.summary_plot(sv_c2, X_m2, feature_names=FEAT_M2, show=False)
plt.gcf().set_size_inches(9, 6)
plt.title('SHAP beeswarm — C2 vs C1+C3', fontsize=11)
plt.tight_layout()
plt.show()

## 5. modelo 2: comparação multi-modelo (3 classes)

mesmos 6 modelos aplicados ao problema original multiclasse (C1/C2/C3).  
métrica principal: F1-weighted (lida com desbalanceamento entre clusters).

In [ ]:
# XGBoost multiclasse precisa de target numerico
le = LabelEncoder()
y_m2_enc = le.fit_transform(y_m2)  # C1->0, C2->1, C3->2

MODELS_M2 = {
    'Logistic Regression': LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=1000,
        multi_class='multinomial', random_state=SEED),
    'SVM linear': SVC(
        kernel='linear', C=1.0, class_weight='balanced',
        probability=True, random_state=SEED),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5, class_weight='balanced', random_state=SEED),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced', random_state=SEED, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1, random_state=SEED),
    'XGBoost': XGBClassifier(
        n_estimators=400, max_depth=3, learning_rate=0.05,
        eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0),
}

SCORING_M2 = {'f1_weighted': 'f1_weighted', 'f1_macro': 'f1_macro'}

results_m2 = {}
for name, model in MODELS_M2.items():
    # usar y_m2 (strings) para todos exceto XGBoost, que precisa de inteiros
    y_target = y_m2_enc if name == 'XGBoost' else y_m2
    cv_res = cross_validate(model, X_m2, y_target, cv=cv5,
                            scoring=SCORING_M2, n_jobs=-1)
    results_m2[name] = {
        metric: {'mean': float(cv_res[f'test_{metric}'].mean()),
                 'std' : float(cv_res[f'test_{metric}'].std())}
        for metric in SCORING_M2
    }
    print(f'  {name:<22}  f1_weighted={results_m2[name]["f1_weighted"]["mean"]:.3f}'
          f'  f1_macro={results_m2[name]["f1_macro"]["mean"]:.3f}')

In [ ]:
metrics_m2 = list(SCORING_M2)
means_m2 = pd.DataFrame(
    {m: [results_m2[mod][m]['mean'] for mod in results_m2] for m in metrics_m2},
    index=list(results_m2)
)
stds_m2 = pd.DataFrame(
    {m: [results_m2[mod][m]['std'] for mod in results_m2] for m in metrics_m2},
    index=list(results_m2)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(means_m2, annot=True, fmt='.3f', cmap='YlGn', ax=axes[0],
            linewidths=0.5, vmin=0, vmax=1, annot_kws={'size': 12})
axes[0].set_title('Modelo 2: comparacao multi-modelo (C1/C2/C3)', fontsize=11)

x = np.arange(len(results_m2))
w = 0.35
for i, (metric, color) in enumerate(zip(metrics_m2, ['#378ADD', '#1D9E75'])):
    axes[1].bar(x + i*w, means_m2[metric].values, w, label=metric,
                color=color, alpha=0.85,
                yerr=stds_m2[metric].values, capsize=3)
axes[1].set_xticks(x + w/2)
axes[1].set_xticklabels(list(results_m2), rotation=25, ha='right', fontsize=9)
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('score (CV 5-fold)')
axes[1].set_title('F1-weighted e F1-macro por modelo', fontsize=11)
axes[1].legend()

# linha de referencia: RF sprint 2
axes[1].axhline(0.5510, color='gray', linestyle='--', linewidth=1,
                label='RF sprint 2 (F1=0.551)')
axes[1].legend(fontsize=8)

plt.suptitle('Modelo 2: comparacao multi-modelo (cluster DMM, n=412)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

best_m2 = means_m2['f1_weighted'].idxmax()
print(f'melhor modelo (F1-weighted): {best_m2}  →  {means_m2.loc[best_m2,"f1_weighted"]:.3f}')
print(f'baseline sprint 2 (RF)     : 0.551')

## 6. exportar resultados em JSON

todos os experimentos desta sprint salvos em um único arquivo para consulta centralizada e comparação futura.

In [ ]:
sprint5_results = {
    'modelo1_multi_modelo': {
        'descricao': 'comparacao de 6 modelos — desfecho adverso (n=208, 12 features)',
        'metricas': ['recall', 'roc_auc', 'f1'],
        'cv': '5-fold estratificada',
        'resultados': results_m1,
    },
    'modelo1_ablacao': {
        'descricao': 'ablacao de features com XGBoost — desfecho adverso',
        'resultados': ablation_results,
    },
    'modelo2_c2_binario': {
        'descricao': 'C2 vs C1+C3 — problema binario, 6 modelos, n=412',
        'metricas': ['recall', 'roc_auc', 'f1'],
        'resultados': results_c2,
    },
    'modelo2_multi_modelo': {
        'descricao': 'comparacao de 6 modelos — cluster DMM (C1/C2/C3, n=412)',
        'metricas': ['f1_weighted', 'f1_macro'],
        'referencia_sprint2': {'rf_f1_weighted': 0.5510},
        'resultados': results_m2,
    },
    'novas_features': {
        'descricao': 'features extraidas de State e baby_birthday',
        'features_criadas': NEW_FEAT_COLS,
        'impacto_ablacao': ablation_results.get('+ features novas', {}),
    },
}

out_path = '../data/processed/sprint5_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(sprint5_results, f, indent=2, ensure_ascii=False)

size_kb = os.path.getsize(out_path) / 1024 if hasattr(__builtins__, '__import__') else 0
print(f'resultados salvos em {out_path}')
print()
print('=== resumo final ===')
print()
print('Modelo 1 (desfecho adverso) — recall por modelo:')
for mod, res in sorted(results_m1.items(), key=lambda x: -x[1]['recall']['mean']):
    print(f'  {mod:<22}  recall={res["recall"]["mean"]:.3f}  auc={res["roc_auc"]["mean"]:.3f}')

print()
print('Ablacao (XGBoost) — recall por subset:')
for sub, res in ablation_results.items():
    delta = res['recall']['mean'] - ablation_results['todas (baseline)']['recall']['mean']
    print(f'  {sub:<30}  recall={res["recall"]["mean"]:.3f}  delta={delta:+.3f}')

print()
print('C2 vs C1+C3 — AUC por modelo:')
for mod, res in sorted(results_c2.items(), key=lambda x: -x[1]['roc_auc']['mean']):
    print(f'  {mod:<22}  auc={res["roc_auc"]["mean"]:.3f}  recall={res["recall"]["mean"]:.3f}')

print()
print('Modelo 2 (cluster DMM) — F1-weighted por modelo:')
for mod, res in sorted(results_m2.items(), key=lambda x: -x[1]['f1_weighted']['mean']):
    print(f'  {mod:<22}  f1_weighted={res["f1_weighted"]["mean"]:.3f}')

## 7. conclusoes

### 7.1 qual modelo é mais robusto?
o heatmap de comparação responde: qual modelo mantém o recall mais alto de forma consistente entre os folds? se o XGBoost da sprint 3 se confirmar como melhor ou similar ao Random Forest e Gradient Boosting, a escolha da sprint 3 é validada. se outro modelo superar, é candidato à substituição.

### 7.2 o que muda ao remover AMR e VF?
a ablação quantifica o valor do sequenciamento metagenômico: o delta de recall entre 'todas' e 'sem amr + vf' é o preço de não ter esse exame. se o delta for pequeno (< 0.05), as features clínicas simples já capturam a maior parte do sinal — o modelo tem valor de triagem sem exame especializado. se for grande (> 0.10), o sequenciamento é indispensável.

### 7.3 C2 é separável?
se nenhum modelo supera AUC ~ 0.65 no problema C2 vs C1+C3, a conclusão é que **C2 é estruturalmente inacessível com features clínicas** — sua definição depende de composição bacteriana granular que as variáveis de nascimento não capturam. isso valida a decisão de usar o cluster como target de pesquisa, não como ferramenta de triagem.

### 7.4 as novas features (estação, estado) ajudam?
se a linha '+ features novas' no heatmap de ablação mostrar melhora, sazonalidade e localização geográfica carregam sinal biológico — provável via microbioma ambiental e padrões de dieta regional. se não, a inclusão não é justificada pelo custo de coleta.